# 66 — LGBM: CRC + ChEMBL PXR-Direct (same target, weight=0.8)

Only ChEMBL records mapped directly to PXR (CHEMBL3401). Highest-confidence external source — same protein, different assay formats.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED,
    verbose=-1, n_jobs=4,
)


In [2]:
def full_metrics(y_true, y_pred, cliff_pairs_df=None, label=""):
    """RAE, MAE, R², Pearson, Spearman, Kendall, Cliff_accuracy."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    mae_v  = float(np.mean(np.abs(yt - yp)))
    rae_v  = mae_v / float(np.mean(np.abs(yt - yt.mean()))) if yt.std() > 0 else float("nan")
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2_v   = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    pr_v, _ = stats.pearsonr(yt, yp)
    sp_v, _ = stats.spearmanr(yt, yp)
    kt_v, _ = stats.kendalltau(yt, yp)

    m = dict(RAE=rae_v, MAE=mae_v, R2=r2_v,
             Pearson=pr_v, Spearman=sp_v, Kendall=kt_v)

    if cliff_pairs_df is not None and len(cliff_pairs_df) > 0:
        correct = total = 0
        for _, row in cliff_pairs_df.iterrows():
            ia, ii = int(row.get("idx_active", -1)), int(row.get("idx_inactive", -1))
            if 0 <= ia < len(yp) and 0 <= ii < len(yp):
                correct += int(yp[ia] > yp[ii])
                total   += 1
        m["Cliff_acc"] = correct / total if total else float("nan")

    if label:
        cliff_str = f"  Cliff_acc={m.get('Cliff_acc', float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f}  MAE={mae_v:.4f}  R²={r2_v:.4f}  "
              f"Pearson={pr_v:.4f}  Spearman={sp_v:.4f}  Kendall={kt_v:.4f}{cliff_str}")
    return m


In [3]:
tr = load_train()
te = load_test()
print(f"CRC train: {len(tr):,}  |  Test: {len(te):,}")

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
active_mask = y_tr >= 5.5
print(f"X_tr: {X_tr.shape}  actives: {active_mask.sum()}")

cliff_pairs = (pd.read_parquet(DATA_PROCESSED / "cliff_pairs.parquet")
               if (DATA_PROCESSED / "cliff_pairs.parquet").exists()
               else pd.DataFrame())
print(f"Cliff pairs available: {len(cliff_pairs)}")


CRC train: 4,139  |  Test: 513


X_tr: (4139, 2265)  actives: 380
Cliff pairs available: 149


In [4]:
def run_cv(X_int, y_int, splits, X_ext=None, y_ext=None, w_ext=None,
           label="", params=LGBM_PARAMS):
    oof = np.full(len(y_int), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        Xf = X_int[tr_idx]; yf = y_int[tr_idx]
        Xv = X_int[va_idx]; yv = y_int[va_idx]
        wf = np.ones(len(yf), dtype=np.float32)
        if X_ext is not None and len(X_ext) > 0:
            Xf = np.vstack([Xf, X_ext])
            yf = np.concatenate([yf, y_ext])
            wf = np.concatenate([wf, w_ext if w_ext is not None
                                  else np.ones(len(y_ext), dtype=np.float32)])
        m = lgb.train(params, lgb.Dataset(Xf, label=yf, weight=wf),
                      valid_sets=[lgb.Dataset(Xv, label=yv)],
                      callbacks=[lgb.early_stopping(50, verbose=False),
                                 lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(Xv)
        print(f"  fold {fold+1}  val_RAE={rae(yv, oof[va_idx]):.4f}", flush=True)
    m_all    = full_metrics(y_int, oof, cliff_pairs, label=label)
    m_active = full_metrics(y_int[active_mask], oof[active_mask],
                            label=f"{label} [active≥5.5]")
    return oof, m_all, m_active


def train_final_and_predict(X_tr_all, y_tr_all, w_tr_all, X_te, params=LGBM_PARAMS):
    m = lgb.train(params, lgb.Dataset(X_tr_all, label=y_tr_all, weight=w_tr_all),
                  callbacks=[lgb.log_evaluation(-1)])
    return np.clip(m.predict(X_te), y_tr_all.min() - 0.5, y_tr_all.max() + 0.5)


In [5]:
ext_df = pd.read_parquet(DATA_EXTERNAL / "chembl_nr_extended.parquet")
ext_pxr = ext_df[ext_df["target_name"] == "PXR"].copy()
# Also include newly fetched all-types if available
pxr_all_path = DATA_EXTERNAL / "chembl_pxr_all_types.parquet"
if pxr_all_path.exists():
    extra = pd.read_parquet(pxr_all_path)
    extra["target_name"] = "PXR"
    ext_pxr = pd.concat([ext_pxr, extra], ignore_index=True)

ext_pxr = ext_pxr.dropna(subset=["smiles","pec50"])
ext_pxr["std_smi"] = ext_pxr["smiles"].map(
    lambda s: __import__("pxr.chem",fromlist=["standardize_smiles"]).standardize_smiles(s))
ext_pxr = ext_pxr.dropna(subset=["std_smi"]).drop_duplicates(subset=["std_smi"])
# Remove train overlaps
from pxr.chem import to_inchikey
tr_iks = set(tr["smiles"].map(to_inchikey))
ext_pxr["ik"] = ext_pxr["std_smi"].map(to_inchikey)
ext_pxr = ext_pxr[~ext_pxr["ik"].isin(tr_iks)]
print(f"ChEMBL PXR-direct (novel): {len(ext_pxr):,} rows  "
      f"pEC50 range [{ext_pxr['pec50'].min():.1f}, {ext_pxr['pec50'].max():.1f}]")

X_ext = impute(combined(ext_pxr["std_smi"].tolist()))
y_ext = ext_pxr["pec50"].clip(3.5, 9.0).values.astype(np.float64)
W_EXT = 0.8
w_ext = np.full(len(y_ext), W_EXT, dtype=np.float32)


ChEMBL PXR-direct (novel): 901 rows  pEC50 range [4.0, 8.6]


In [6]:
print("Running scaffold 5-fold CV...")
oof, m_all, m_active = run_cv(
    X_tr, y_tr, splits,
    X_ext=X_ext if len(X_ext) > 0 else None,
    y_ext=y_ext if len(X_ext) > 0 else None,
    w_ext=w_ext if len(X_ext) > 0 else None,
    label="CRC+ChEMBL_PXR"
)
print(f"\nAugmented with {len(X_ext):,} external rows (weight scale={W_EXT:.2f})" if len(X_ext) > 0
      else "\nNo external augmentation (data empty)")

results_df = pd.DataFrame([m_all, m_active], index=["overall", "active≥5.5"])
print("\n" + results_df.round(4).to_string())


Running scaffold 5-fold CV...


  fold 1  val_RAE=0.5037


  fold 2  val_RAE=0.5870


  fold 3  val_RAE=0.6053


  fold 4  val_RAE=0.5689


  fold 5  val_RAE=0.6053


  [CRC+ChEMBL_PXR] RAE=0.5692  MAE=0.5179  R²=0.5877  Pearson=0.7687  Spearman=0.7196  Kendall=0.5297  Cliff_acc=nan
  [CRC+ChEMBL_PXR [active≥5.5]] RAE=3.4316  MAE=0.7196  R²=-8.0771  Pearson=0.1805  Spearman=0.1302  Kendall=0.0892

Augmented with 901 external rows (weight scale=0.80)

               RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
overall     0.5692  0.5179  0.5877   0.7687    0.7196   0.5297        NaN
active≥5.5  3.4316  0.7196 -8.0771   0.1805    0.1302   0.0892        NaN


In [7]:
# Final model on all data
w_base = np.ones(len(y_tr), dtype=np.float32)
if len(X_ext) > 0:
    X_all = np.vstack([X_tr, X_ext])
    y_all = np.concatenate([y_tr, y_ext])
    w_all = np.concatenate([w_base, w_ext])
else:
    X_all, y_all, w_all = X_tr, y_tr, w_base

te_preds = train_final_and_predict(X_all, y_all, w_all, X_te)
np.save(DATA_PROCESSED / "oof_lgbm_chembl_pxr_direct.npy", oof)
np.save(DATA_PROCESSED / "te_oof_lgbm_chembl_pxr_direct.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub) == 513 and sub["pEC50"].notna().all()
out = SUBMISSIONS / "66_lgbm_chembl_pxr_direct.csv"
sub.to_csv(out, index=False)
print(f"Saved {out}")
print(f"Test preds  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}")



Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\66_lgbm_chembl_pxr_direct.csv
Test preds  min=2.32  median=4.97  max=6.15
